# Motion denoising: PoseD-Flow's protocol (their Tab. 3), RSGM against DPoser and PoseD-Flow

Noisy 3D joints of a motion are cleaned by fitting the poses of a body to them with a pose prior. Everything follows PoseD-Flow (Nadar, Foti, Birdal, CVPR 2026; their `motion_denoising.py`) and DPoser, `hp/evaluation/denoising.py`:

- **Data**: the **test** split is their HumanEva set of AMASS: 190 chunks of 60 frames at 30 Hz (SMPL-X fits, the first and last 10% of a motion cut). The **validation** split is 100 chunks drawn with seed 0 from HDM05, SFU and MoSh; settings are chosen there, the test is run once per row.
- **Noise**: Gaussian, σ = 40 or 100 mm, on the 22 body joints of SMPL-X neutral (betas 0); fixed per sequence.
- **Metrics** (mm): MPJPE of the 22 joints (their table), MPVPE of all vertices with the toes at rest (added), and the MPJPE of the noisy input.
- **Methods**: **D-Flow** (PoseD-Flow's guidance through the ODE, their RiemannianAdam, lr 0.25, Euler 100) and **DPoser's optimization**: Adam on the poses with a data term, a temporal term and the prior term, which for RSGM is score distillation on SO(3)^J (`hp.models.motion_denoising.dposer_denoise`). The settings of RSGM in DPoser's optimization were tuned on validation (Optuna, `config/denoising_tuning/`); the others are the authors'.

Nothing is computed here: the rows are read from the caches of `scripts/evaluate_denoising.py` (and `explore/evaluation/denoising_aniso/rows.py`). A row whose cached file was made with other settings than its current spec (e.g. before the tuning) is labelled **[other settings]** and hidden unless `SHOW_OTHER`.

In [ ]:
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
for path in (ROOT / "src", ROOT / "scripts", ROOT / "explore" / "evaluation" / "denoising_aniso"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np

import evaluate_denoising as ed
import rows as aniso_rows  # noqa: F401  (adds the rows of the per-joint-noise models to ed.MODELS)
from hp.evaluation import denoising_results as R
from hp.evaluation.denoising import PAPER, SIGMAS

assert R.__file__.startswith(str(ROOT / "src")), f"hp comes from {R.__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
RESULTS = Path("~/data/experiments/denoising").expanduser()
# the Mac cache first, then copies from the servers (cache_<server>, e.g. the validation rows run there)
CACHES = [RESULTS / "cache"] + sorted(p for p in RESULTS.glob("cache_*") if p.is_dir())
SHOW_OTHER = False                                   # rows cached with other settings than their current spec
REFERENCE = "DPoser: their full, DPoser method"      # the row the others are compared with (section 3)
OURS = "RSGM 1536x6: their c10, DPoser method"       # "ours" by the rule of rsgm_sds_40mm.yaml (lowest validation MPJPE)
# the rows shown, in this order within their group (section 1); a commented line hides its row. The rows of
# rows.py are named with "_" (abl_residual), and their cache folders are those of the same rows of ed.MODELS
ROWS = [
    "w/o prior",
    "PoseD-Flow: their checkpoint",
    "DPoser: their full, DPoser method",
    "DPoser: their full, DPoser method, tuned",          # tuned like RSGM (ed.DPOSER_TUNED)
    "DPoser: their full, D-Flow",
    # RSGM at the settings tuned on validation (ed.SDS)
    "RSGM 1536x6: their c10, DPoser method",
    "RSGM abl_residual: their c10, DPoser method",
    "RSGM 512x4: their c10, DPoser method",
    "RSGM 512x4: their full, DPoser method",
    "RSGM res_norm: their c10, DPoser method",
    "RSGM abl residual lik: their c10, DPoser method",       # validation only
    "RSGM abl residual ln m3: their c10, DPoser method",     # validation only
    "RSGM abl residual ln m4: their c10, DPoser method",     # validation only
    "RSGM 512x4: their c10, DPoser method, DPoser's steps",  # the tuned settings at DPoser's number of steps
    "RSGM 512x4: their c10, D-Flow",
    # the first guess of the settings (before the tuning; test run by mistake, see the report)
    "RSGM 1536x6: their c10, DPoser method, first guess",
    "RSGM 512x4: their c10, DPoser method, first guess",
    # "RSGM 512x4: their full, DPoser method, first guess",
    # "RSGM res_norm: their c10, DPoser method, first guess",
    # per-joint noise (validation only, cache_aniso; explore/evaluation/denoising_aniso/rows.py)
    "RSGM abl_norm: their c10, DPoser method",
    "RSGM abl_residual_aniso: their c10, DPoser method, t geo",
    "RSGM abl_norm_aniso: their c10, DPoser method, t geo",
    # "RSGM abl_residual_aniso: their c10, DPoser method",   # the isotropic t: noise it was not trained on
    # "RSGM abl_norm_aniso: their c10, DPoser method",
    # "RSGM abl_norm_aniso: their c10, DPoser method, t max",
]
SHORT_NAMES = {                                      # in the plots and pictures; the others keep their labels
    "w/o prior": "w/o prior",
    "PoseD-Flow: their checkpoint": "PoseD-Flow",
    "DPoser: their full, DPoser method": "DPoser",
    "DPoser: their full, DPoser method, tuned": "DPoser, tuned",
    "DPoser: their full, D-Flow": "DPoser + D-Flow",
    "RSGM 1536x6: their c10, DPoser method": "RSGM 1536x6 + SDS (ours)",
    "RSGM abl_residual: their c10, DPoser method": "RSGM abl residual + SDS",
    "RSGM 512x4: their c10, DPoser method": "RSGM 512x4 + SDS",
    "RSGM 512x4: their full, DPoser method": "RSGM 512x4 full + SDS",
    "RSGM res_norm: their c10, DPoser method": "RSGM res_norm + SDS",
    "RSGM 512x4: their c10, DPoser method, DPoser's steps": "RSGM 512x4 + SDS, DPoser's steps",
    "RSGM 512x4: their c10, D-Flow": "RSGM 512x4 + D-Flow",
    "RSGM 1536x6: their c10, DPoser method, first guess": "RSGM 1536x6 + SDS, first guess",
}
# ----------------------------------------------------------------------------------------

records = R.load(CACHES, models=ed.MODELS)
records = [r for r in records if SHOW_OTHER or not r["label"].endswith("[other settings]")]
hidden = sorted({r["label"] for r in records if r["label"] not in ROWS and not r["label"].endswith("[other settings]")})
records = [r for r in records if r["label"] in ROWS or r["label"].endswith("[other settings]")]
if hidden:
    print(f"hidden (not in ROWS): {len(hidden)} rows")
print(f"{len(records)} cached (row, split, noise level) from {', '.join(p.name for p in CACHES)}")

## 1. The table, next to their Tab. 3

Per split, every cached row: MPJPE / MPVPE at 40 and 100 mm (mm, means over all frames of all sequences), and the MPJPE of the noisy input. Their rows (AMASS, MPJPE only) above ours. The test cells of a row are empty until it has been run on the test.

In [ ]:
def order(label):
    # the rows in groups: no prior, the baselines, RSGM; within a group in the order of ROWS
    group = 0 if label == "w/o prior" else 1 if not label.startswith("RSGM") else 2
    return group, ROWS.index(label) if label in ROWS else len(ROWS), label


def table(split):
    rows = sorted({r["label"] for r in records if r["split"] == split}, key=order)
    head = f"{'':58s}" + "".join(f"{f'{round(s * 1000)} mm: MPJPE / MPVPE':>27s}" for s in SIGMAS) + f"{'input':>8s}"
    print(f"\n{split} ({'190 HumanEva chunks' if split == 'test' else '100 chunks of HDM05, SFU, MoSh'})\n{head}")
    if split == "test":
        for name, v in PAPER.items():
            print(f"{name + ' (paper)':58s}" + "".join(f"{v[i]:>15.2f}{'':12s}" for i in range(len(SIGMAS))))
    for label in rows:
        cells, inp = [], []
        for sigma in SIGMAS:
            r = R.find(records, label, sigma, split)
            cells.append(f"{'-':>27s}" if r is None else f"{r['mpjpe']:>15.2f} / {r['mpvpe']:6.2f}{'':3s}")
            if r is not None:
                inp.append(f"{r['input_mpjpe']:.0f}")
        mark = "  <- ours" if label == OURS else ""
        print(f"{label[:57]:58s}" + "".join(cells) + f"{'/'.join(inp):>8s}{mark}")


for split in ("valid", "test"):
    table(split)
print("\nmm; input: MPJPE of the noisy joints at 40 / 100 mm. Validation chose the settings; the test only reports.")

## 2. Is the difference real? Paired over the sequences


Every row is run on the same sequences with the same noise, so two rows are compared sequence by sequence: the mean difference of the per-sequence MPJPE (row − reference; negative: the row is better), its 95% bootstrap interval over the sequences, and the share of sequences where the row is better. An interval that contains 0 means the sequences do not tell the two rows apart. The interval covers the choice of sequences, not the seed of the training or of the optimization.

In [ ]:
def paired_table(split, reference=REFERENCE, metric="mpjpe"):
    print(f"\n{split}, {metric.upper()}: row - {reference}")
    print(f"{'':58s}" + "".join(f"{f'{round(s * 1000)} mm: diff [95% CI]  better':>36s}" for s in SIGMAS))
    for label in sorted({r["label"] for r in records if r["split"] == split}, key=order):
        if label == reference:
            continue
        cells = []
        for sigma in SIGMAS:
            a, b = R.find(records, label, sigma, split), R.find(records, reference, sigma, split)
            if a is None or b is None or a["names"] != b["names"]:
                cells.append(f"{'-':>36s}")
                continue
            p = R.paired(a["per_sequence"][metric], b["per_sequence"][metric])
            cells.append(f"{p['mean']:+7.2f} [{p['lo']:+6.2f}, {p['hi']:+6.2f}] {100 * p['a_better']:6.0f}%")
        print(f"{label[:57]:58s}" + "".join(f"{c:>36s}" for c in cells))


for split in ("valid", "test"):
    paired_table(split)

## 3. Per dataset (validation) and per subject (test)

The mean MPJPE of a row over the sequences of each group. A row that wins on average but loses on one group is worth a look in the pictures of section 5.

In [ ]:
SIGMA = 0.1   # the noise level of this section and the next


def group_table(split, sigma=SIGMA, metric="mpjpe"):
    shown = [r for r in (R.find(records, label, sigma, split)
                         for label in sorted({r["label"] for r in records if r["split"] == split}, key=order)) if r]
    if not shown:
        return
    groups = list(R.by_group(shown[0], metric))
    counts = {g: sum(R.dataset_of(n) == g for n in shown[0]["names"]) for g in groups}
    print(f"\n{split}, {round(sigma * 1000)} mm, {metric.upper()} per group (sequences)")
    print(f"{'':58s}" + "".join(f"{f'{g} ({counts[g]})':>14s}" for g in groups))
    for r in shown:
        values = R.by_group(r, metric)
        print(f"{r['label'][:57]:58s}" + "".join(f"{values[g]:14.2f}" for g in groups))


for split in ("valid", "test"):
    group_table(split)

## 4. The error over the frames of a chunk

The MPJPE per frame, averaged over the sequences. The temporal term pulls each frame toward its neighbours, which the first and the last frame have on one side only: a rise at the ends is the price of the chunks of 60 frames, the same for every row.

In [ ]:
import matplotlib.pyplot as plt
import torch

SPLIT = "valid"
LABELS = [REFERENCE, OURS, "PoseD-Flow: their checkpoint", "w/o prior"]

with plt.style.context("default"):
    fig, axes = plt.subplots(1, len(SIGMAS), figsize=(6 * len(SIGMAS), 3.4))
    for ax, sigma in zip(np.atleast_1d(axes), SIGMAS):
        for label in LABELS:
            r = R.find(records, label, sigma, SPLIT)
            if r is None:
                continue
            per_frame = torch.load(r["file"], weights_only=True)["metrics"]["mpjpe"].double().numpy().mean(0)
            ax.plot(per_frame, lw=1.5, label=SHORT_NAMES.get(label, label))
        ax.set_title(f"{SPLIT}, {round(sigma * 1000)} mm", fontsize=10)
        ax.set_xlabel("frame")
        ax.set_ylabel("MPJPE (mm)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7, frameon=False)
    plt.tight_layout()
    plt.show()

## 5. Pictures of single sequences


Read from the caches: nothing is recomputed. A line for the ground truth (blue) with the **noisy joints** (red dots), then one line per method: its denoised body at a few frames, the vertices coloured by their distance to the ground truth (light: right, dark: 100 mm or more), its MPJPE and MPVPE on this sequence on the left. Below: the joint error of every method and of the input over the 60 frames.

- **split**, **noise**; **methods**: how many, and one drop-down per slot (with its means on this split and noise level). Method 1 and 2 are the ones **pick** looks at.
- **pick sequence**: *random*; *hard for 1* / *easy for 1*: one of the `TOP` sequences with the highest / lowest MPJPE of method 1; *1 better than 2* / *2 better than 1*: one of the `TOP` where method 1 is most below / above method 2. Then the picture is drawn.
- **frames**: how many, evenly spread over the chunk; **view**: front or side; **Show** draws the sequence in the box; **Save**: the figure as .png and .pdf in `SAVE_DIR`.


In [ ]:
import traceback

import ipywidgets as widgets
from IPython.display import display

from hp.evaluation.denoising import Body
from hp.visualization.completion_figure import PICKS, pick_pose
from hp.visualization.denoising_figure import denoising_case, denoising_figure, per_frame_errors, render_case

# ---- parameters ----------------------------------------------------------------------
N_COMPARE = 3                                     # methods, one per slot (method 1 is the one "pick" looks at)
DEFAULT_METHODS = [OURS, REFERENCE, "PoseD-Flow: their checkpoint"]
TOP = 10                                          # "hard for 1" etc. pick among the TOP sequences
SAVE_DIR = "~/data/denoising_figures"
SIZE = (260, 380)
# ----------------------------------------------------------------------------------------

body = Body()
rng = np.random.default_rng()
L = lambda w: widgets.Layout(width=w)


def labels_of(split, sigma):
    return sorted({r["label"] for r in records if r["split"] == split and r["sigma"] == sigma}, key=order)


def option(label, split, sigma):
    # the text of a method in the drop-downs: its means over the sequences of this split and noise level, then its
    # label; the numbers first, padded with figure spaces (as wide as a digit), as in pose_completion.ipynb
    r = R.find(records, label, sigma, split)
    pad = lambda x: format(x, "6.2f").replace(" ", "\u2007")
    return f"MPJPE {pad(r['mpjpe'])}  ·  MPVPE {pad(r['mpvpe'])}   {label}"


split_box = widgets.ToggleButtons(options=["valid", "test"], value="test", description="split")
sigma_box = widgets.ToggleButtons(options=[("40 mm", 0.04), ("100 mm", 0.1)], value=0.1, description="noise")
count_box = widgets.BoundedIntText(value=N_COMPARE, min=1, max=8, description="methods", layout=L("160px"))
methods_area = widgets.VBox()
method_boxes = []
sequence_box = widgets.BoundedIntText(value=0, min=0, max=189, description="sequence", layout=L("180px"))
pick_box = widgets.ToggleButtons(options=PICKS, value="random", description="pick", style={"button_width": "120px"})
pick_button = widgets.Button(description="pick sequence", icon="random")
frames_box = widgets.BoundedIntText(value=5, min=1, max=12, description="frames", layout=L("160px"))
view_box = widgets.ToggleButtons(options=[("front", 0), ("side", 90)], value=0, description="view")
curves_box = widgets.Checkbox(value=True, description="error curves")
show_button = widgets.Button(description="Show", button_style="primary", icon="play")
save_button = widgets.Button(description="Save", icon="save")
out, msg = widgets.Output(), widgets.Output()
last = {}


def say(text):
    with msg:
        msg.clear_output(wait=True)
        print(text)


def shown_labels():
    return labels_of(split_box.value, sigma_box.value)


def set_count(change=None):
    # as many drop-downs as count_box says; those there keep their choice
    labels, n = shown_labels(), count_box.value
    kept = [box.value for box in method_boxes if box.value in labels][:n]
    rest = [m for m in DEFAULT_METHODS + labels if m in labels and m not in kept]
    values = kept + rest[:n - len(kept)]
    options = [(option(m, split_box.value, sigma_box.value), m) for m in labels]
    method_boxes[:] = [widgets.Dropdown(options=options, value=v, description=f"method {i + 1}", layout=L("900px"))
                       for i, v in enumerate(values)]
    methods_area.children = method_boxes


def set_split(change=None):
    # the drop-downs follow the split and the noise level (their means; a method not cached there is replaced)
    set_count()
    n = len(next((r["names"] for r in records if r["split"] == split_box.value), [None] * 190))
    sequence_box.max = n - 1


def methods():
    return list(dict.fromkeys(box.value for box in method_boxes))   # in order, without repeats


def pick(_=None):
    try:
        errors = per_frame_errors(records, methods()[:2], split_box.value, sigma_box.value)
        sequence_box.value = pick_pose(pick_box.value, errors, rng, top=TOP)
    except ValueError as error:
        return say(str(error))
    show()


def show(_=None):
    split, sigma, sequence = split_box.value, sigma_box.value, sequence_box.value
    say(f"rendering {split}, {round(sigma * 1000)} mm, sequence {sequence} ...")
    try:
        frames = sorted({int(round(f)) for f in np.linspace(0, 59, frames_box.value)})
        case = denoising_case(records, methods(), split, sigma, sequence, body)
        images = render_case(case, body, frames=frames, azimuth=view_box.value, size=SIZE)
        fig = denoising_figure(case, images, names=SHORT_NAMES, curves=curves_box.value)
    except Exception:
        return say(traceback.format_exc())
    last.update(fig=fig, name=f"denoising_{split}_{round(sigma * 1000)}mm_seq{sequence}")
    with out:
        out.clear_output(wait=True)
        display(fig)
    plt.close(fig)
    width = max(len(SHORT_NAMES.get(m["label"], m["label"])) for m in case["methods"])
    lines = [f"  {SHORT_NAMES.get(m['label'], m['label']):<{width}}  MPJPE {m['mpjpe'].mean():6.2f}  "
             f"MPVPE {m['mpvpe'].mean():6.2f} mm" for m in case["methods"]]
    say(f"{case['name']} ({split}, sequence {sequence}, {round(sigma * 1000)} mm; input MPJPE "
        f"{case['input'].mean():.1f} mm):\n" + "\n".join(lines))


def save(_=None):
    if "fig" not in last:
        return say("nothing shown yet")
    folder = Path(SAVE_DIR).expanduser()
    folder.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "pdf"):
        last["fig"].savefig(folder / f"{last['name']}.{suffix}", dpi=200, bbox_inches="tight")
    say(f"saved {folder / last['name']}.png / .pdf")


set_split()
count_box.observe(set_count, names="value")
split_box.observe(set_split, names="value")
sigma_box.observe(set_split, names="value")
pick_button.on_click(pick)
show_button.on_click(show)
save_button.on_click(save)
display(widgets.VBox([
    widgets.HBox([split_box, sigma_box]),
    count_box, methods_area,
    widgets.HBox([sequence_box, pick_button]), pick_box,
    widgets.HBox([frames_box, view_box, curves_box]),
    widgets.HBox([show_button, save_button]),
    msg, out,
]))

In [ ]:
# A fixed figure for the paper: one sequence, the same methods; edit and run
FIGURE = dict(split="test", sigma=0.1, sequence=12, frames=(0, 15, 30, 45, 59), view=0,
              methods=[REFERENCE, OURS, "PoseD-Flow: their checkpoint"], file="denoising_test_100mm")

case = denoising_case(records, FIGURE["methods"], FIGURE["split"], FIGURE["sigma"], FIGURE["sequence"], body)
fig = denoising_figure(case, render_case(case, body, frames=FIGURE["frames"], azimuth=FIGURE["view"], size=SIZE),
                       names=SHORT_NAMES)
folder = Path(SAVE_DIR).expanduser()
folder.mkdir(parents=True, exist_ok=True)
for suffix in ("png", "pdf"):
    fig.savefig(folder / f"{FIGURE['file']}.{suffix}", dpi=200, bbox_inches="tight")
print(f"saved {folder / FIGURE['file']}.png / .pdf")